<a href="https://colab.research.google.com/github/fanisam/Praktikum_AI/blob/main/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pada studi kasus ini dilakukan klasifikasi jenis wine menggunakan metode Artificial Neural Network (ANN). Dataset yang digunakan berisi data karakteristik kimia dari berbagai jenis wine, seperti kadar alcohol, magnesium, flavanoids, dan beberapa fitur lainnya. Setiap data wine memiliki label class yang menunjukkan jenis wine tertentu. Tujuan dari study case ini adalah membuat model ANN yang dapat mempelajari pola dari data tersebut sehingga mampu memprediksi jenis wine secara otomatis berdasarkan karakteristik yang dimiliki.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Load dataset
df = pd.read_csv("Wine dataset.csv")

print(df.head())

   class  Alcohol  Malic acid   Ash  Alcalinity of ash  Magnesium  \
0      1    14.23        1.71  2.43               15.6        127   
1      1    13.20        1.78  2.14               11.2        100   
2      1    13.16        2.36  2.67               18.6        101   
3      1    14.37        1.95  2.50               16.8        113   
4      1    13.24        2.59  2.87               21.0        118   

   Total phenols  Flavanoids  Nonflavanoid phenols  Proanthocyanins  \
0           2.80        3.06                  0.28             2.29   
1           2.65        2.76                  0.26             1.28   
2           2.80        3.24                  0.30             2.81   
3           3.85        3.49                  0.24             2.18   
4           2.80        2.69                  0.39             1.82   

   Color intensity   Hue  OD280/OD315 of diluted wines  Proline   
0             5.64  1.04                          3.92      1065  
1             4.38  1.05

In [3]:
# Fitur
X = df.drop('class', axis=1).values

# Target
y = df['class'].values

In [35]:
scaler = StandardScaler()
X = scaler.fit_transform(X)



In [38]:
X = X.T

In [33]:
# Mengubah class menjadi mulai dari 0
y = y - 1

# One hot encoding
Y = np.eye(3)[y].T

In [39]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X.T,
    Y.T,
    test_size=0.2,
    random_state=42
)

# Transpose kembali
X_train = X_train.T
X_test = X_test.T
Y_train = Y_train.T
Y_test = Y_test.T

In [15]:
def initialize_parameters(input_size, hidden_size, output_size):

    np.random.seed(42)

    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }

    return parameters

In [16]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

In [17]:
def softmax(Z):

    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))

    return expZ / np.sum(expZ, axis=0, keepdims=True)

In [18]:
def forward_propagation(X, parameters):

    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]

    # Hidden Layer
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)

    # Output Layer
    Z2 = np.dot(W2, A1) + b2
    A2 = softmax(Z2)

    cache = {
        "Z1": Z1,
        "A1": A1,
        "Z2": Z2,
        "A2": A2
    }

    return A2, cache

In [19]:
def compute_cost(Y, A2):

    m = Y.shape[1]

    cost = -np.sum(Y * np.log(A2 + 1e-8)) / m

    return np.squeeze(cost)

In [20]:
def backward_propagation(X, Y, parameters, cache):

    m = X.shape[1]

    W2 = parameters["W2"]

    dZ2 = cache["A2"] - Y

    dW2 = np.dot(dZ2, cache["A1"].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])

    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {
        "dW1": dW1,
        "db1": db1,
        "dW2": dW2,
        "db2": db2
    }

    return grads

In [21]:
def update_parameters(parameters, grads, learning_rate):

    parameters["W1"] -= learning_rate * grads["dW1"]
    parameters["b1"] -= learning_rate * grads["db1"]

    parameters["W2"] -= learning_rate * grads["dW2"]
    parameters["b2"] -= learning_rate * grads["db2"]

    return parameters

In [22]:
def train_neural_network(
    X,
    Y,
    input_size,
    hidden_size,
    output_size,
    epochs=5000,
    learning_rate=0.01
):

    parameters = initialize_parameters(
        input_size,
        hidden_size,
        output_size
    )

    for i in range(epochs):

        A2, cache = forward_propagation(X, parameters)

        cost = compute_cost(Y, A2)

        grads = backward_propagation(
            X,
            Y,
            parameters,
            cache
        )

        parameters = update_parameters(
            parameters,
            grads,
            learning_rate
        )

        if i % 500 == 0:
            print(f"Epoch {i}, Cost = {cost}")

    return parameters

In [23]:
def predict(X, parameters):

    A2, _ = forward_propagation(X, parameters)

    predictions = np.argmax(A2, axis=0)

    return predictions

In [24]:
trained_parameters = train_neural_network(
    X_train,
    Y_train,
    input_size=13,
    hidden_size=10,
    output_size=3,
    epochs=5000,
    learning_rate=0.01
)

Epoch 0, Cost = 1.09863902813056
Epoch 500, Cost = 0.9391490658541324
Epoch 1000, Cost = 0.18690742540565972
Epoch 1500, Cost = 0.08159666499225972
Epoch 2000, Cost = 0.055571780917176156
Epoch 2500, Cost = 0.04348161683618212
Epoch 3000, Cost = 0.03601101235447372
Epoch 3500, Cost = 0.03075447683753783
Epoch 4000, Cost = 0.026760670050346963
Epoch 4500, Cost = 0.023617957429771515


In [25]:
predictions = predict(X_test, trained_parameters)

actual = np.argmax(Y_test, axis=0)

print("Prediksi:")
print(predictions)

print("Data Asli:")
print(actual)

Prediksi:
[0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]
Data Asli:
[0 0 2 0 1 0 1 2 1 2 0 2 0 1 0 1 1 1 0 1 0 1 1 2 2 2 1 1 1 0 0 1 2 0 0 0]


In [30]:
boolean_result = predictions == actual

print("Boolean Result:")
print(boolean_result)

Boolean Result:
[ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True]


In [27]:
accuracy = np.mean(predictions == actual)

print("Accuracy:", accuracy * 100, "%")

Accuracy: 100.0 %


Hasil klasifikasi menggunakan ANN menunjukkan bahwa model dapat mengenali pola pada Wine Dataset dengan sangat baik sehingga menghasilkan akurasi yang tinggi bahkan mencapai 100%. Hal ini terjadi karena setiap kelas pada dataset memiliki perbedaan fitur yang cukup jelas sehingga mudah dipelajari oleh model. Proses normalisasi data dan penggunaan fungsi aktivasi ReLU serta Softmax juga membantu meningkatkan performa ANN dalam melakukan klasifikasi multi-class. Nilai cost yang terus menurun selama training menandakan bahwa model belajar dengan baik dan error semakin kecil. Secara keseluruhan, ANN berhasil melakukan klasifikasi data wine dengan sangat baik.
